## base setup

In [1]:
#| default_exp core

In [2]:
#| export
import multiprocessing as mp
mp.set_start_method("spawn", force=True)

import os
os.environ["TORCHDYNAMO_DISABLE"] = "1"  # disable Dynamo entirely

import os
import io
import re
import random
import base64
from io import BytesIO

import time
from datetime import timedelta

import numpy as np

import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F

from IPython.display import SVG

from PIL import Image as PILImage

import cv2
import pandas as pd
import json


from diffusers import StableDiffusionPipeline
from transformers import AutoProcessor, AutoModel

import vtracer

import metric

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

## Competition Metric Helpers

We also want to evaluate metrics of the original bitmap before converting to svg. Let’s implement it using [metric package](https://www.kaggle.com/code/jiazhuang/svg-image-fidelity).

In [3]:
import numpy as np
import statistics
import pandas as pd

def image_resize(image, size=(384, 384)):
    return image.convert('RGB').resize(size)

def bitmap_score_instance_impl(multiple_choice_qa, image, random_seed=42):
    rng = np.random.RandomState(random_seed)
    group_seed = rng.randint(0, np.iinfo(np.int32).max)
    image_processor = metric.ImageProcessor(image=image_resize(image), seed=group_seed).apply()
    image = image_processor.image.copy()
    questions = multiple_choice_qa['question']
    choices = multiple_choice_qa['choices']
    answers = multiple_choice_qa['answer']
    aesthetic_score = metric.aesthetic_evaluator.score(image)
    vqa_score = metric.vqa_evaluator.score(questions, choices, answers, image)
    image_processor.reset().apply_random_crop_resize().apply_jpeg_compression(quality=90)
    ocr_score = metric.vqa_evaluator.ocr(image_processor.image)
    instance_score = metric.harmonic_mean(vqa_score, aesthetic_score, beta=0.5) * ocr_score
    return instance_score, vqa_score, ocr_score, aesthetic_score

def bitmap_score_instance(multiple_choice_qa, image, random_seed=42):
    is_single = not isinstance(image, list)
    if is_single:
        multiple_choice_qa = [multiple_choice_qa]
        image = [image]
    
    assert len(multiple_choice_qa) == len(image)

    results = []
    score_df = []
    for one_image, one_multiple_choice_qa in zip(image, multiple_choice_qa, strict=True):
        instance_score, vqa_score, ocr_score, aesthetic_score = bitmap_score_instance_impl(one_multiple_choice_qa, one_image, random_seed=42)
        results.append(instance_score)
        score_df.append([instance_score, vqa_score, ocr_score, aesthetic_score])

    fidelity = statistics.mean(results)
    score_df = pd.DataFrame(score_df, columns=['competition_score', 'vqa_score', 'ocr_score', 'aesthetic_score'])
    if is_single:
        return score_df.iloc[0].to_dict()
    else:
        return float(fidelity), score_df

## Load Data

In [4]:
import pandas as pd
import json
train_df = pd.read_csv('train.csv')
train_question_df = pd.read_parquet('questions.parquet')

train_question_df = train_question_df.groupby('id').apply(lambda df: df.to_dict(orient='list'))
train_question_df = train_question_df.reset_index(name='qa')

train_question_df['question'] = train_question_df.qa.apply(lambda qa: json.dumps(qa['question'], ensure_ascii=False))

train_question_df['choices'] = train_question_df.qa.apply(
    lambda qa: json.dumps(
        [x.tolist() for x in qa['choices']], ensure_ascii=False
    )
)

train_question_df['answer'] = train_question_df.qa.apply(lambda qa: json.dumps(qa['answer'], ensure_ascii=False))

train_df = pd.merge(train_df, train_question_df, how='left', on='id')

train_df['multiple_choice_qa'] = train_df.apply(
    lambda r: {
    'question': json.loads(r.question),
    'choices': json.loads(r.choices),
    'answer': json.loads(r.answer)
    },
    axis=1,
)

# train_df.head()

/tmp/ipykernel_2044/3134480291.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  train_question_df = train_question_df.groupby('id').apply(lambda df: df.to_dict(orient='list'))


## image aug

In [5]:
#| export
from PIL import Image as PILImage
import numpy as np
from skimage.color import rgb2lab, lab2rgb
from skimage import exposure

def enhance_color_richness_lab(image: PILImage.Image, saturation_scale=1.3, contrast_clip=0.001, size=(384, 384)) -> PILImage.Image:
    image = image.resize(size)  # Resize to standard size

    img_np = np.array(image).astype(np.float32) / 255.0
    lab = rgb2lab(img_np)

    L, A, B = lab[..., 0], lab[..., 1], lab[..., 2]

    L_eq = exposure.equalize_adapthist(L / 100.0, clip_limit=contrast_clip) * 100
    A_boosted = A * saturation_scale
    B_boosted = B * saturation_scale

    lab_enhanced = np.stack([L_eq, A_boosted, B_boosted], axis=-1)
    rgb_enhanced = lab2rgb(lab_enhanced)
    rgb_clipped = np.clip(rgb_enhanced, 0, 1)

    return PILImage.fromarray((rgb_clipped * 255).astype(np.uint8))

   

In [6]:
#| export
import re

def add_black_frame(svg_code: str, padding_ratio: float = 0.05) -> str:
    # Ensure viewBox exists or assume default
    match = re.search(r'viewBox="([\d\s\.]+)"', svg_code)
    if match:
        vb = list(map(float, match.group(1).split()))
        x, y, w, h = vb
    else:
        x, y, w, h = 0, 0, 384, 384
        svg_code = svg_code.replace('<svg', f'<svg viewBox="0 0 {w} {h}"', 1)

    # Compute scale and offset
    scale = 1 - 2 * padding_ratio
    tx = w * padding_ratio
    ty = h * padding_ratio

    # Insert a black background
    svg_code = re.sub(
        r'(<svg[^>]*?>)',
        r'\1<rect width="{:.0f}" height="{:.0f}" fill="black"/>'.format(w, h),
        svg_code,
        count=1
    )

    # Wrap everything inside <g transform="...">, preserving any existing <g> and <path>
    content_match = re.search(r'(<g[^>]*>.*?</g>)(.*?</svg>)', svg_code, flags=re.DOTALL)
    if content_match:
        g_block, rest = content_match.groups()
        wrapped = f'<g transform="translate({tx:.1f},{ty:.1f}) scale({scale:.2f})">{g_block}</g>'
        svg_code = re.sub(r'<g[^>]*>.*?</g>', wrapped, svg_code, flags=re.DOTALL)

    return svg_code

## load qa

In [7]:
#| export
from PIL import Image
import ast
import random
import spacy
import pandas as pd

# Load large spaCy model
nlp = spacy.load("en_core_web_lg")

# Extract grouped elements: [descriptive noun phrase] vs [spatial phrase]
def generate_qa_spacy(text):

    try:
        doc = nlp(text)
        descriptive_elements = set(chunk.text for chunk in doc.noun_chunks)
    
        spatial_phrases = set()
        for token in doc:
            if token.dep_ == "prep":
                phrase = token.text
                # Get all children of the preposition (usually includes noun + modifiers)
                object_phrase = " ".join([child.text for child in token.children])
                if object_phrase:
                    phrase = f"{phrase} {object_phrase}"
                spatial_phrases.add(phrase)
    
        questions = []
        for i in list(descriptive_elements):
            question = 'Are there ' + i + ' in the image? Answer yes or no.'
            questions.append(question)
        # for i in list(spatial_phrases):
        #     question = 'Are there something ' + i + '?' 
        #     questions.append(question)

        if not questions:
            return generate_qa(text)
    
        if len(questions) > 2:
            questions = random.sample(questions, 2)

        # add the final question
        q = f"Are there {text}? Answer yes or no."
        questions.append(q)
    
        # Build the QA dictionary
        qa = {
            "question": questions,
            "choices": [["no", "yes"]] * len(questions),
            "answer": ["yes"] * len(questions)
        }
        return qa
        
    except Exception as e:
        print(f"[Prompt fallback] Failed to create qa: {e}")
        return generate_qa(text)


def generate_qa(prompt: str) -> dict:
    """
    Generate VQA-style question-answer dict based on a given prompt.
    Includes yes/no questions and one clarity rating.
    """
    qa = {
        'question': [
            f"Are there {prompt} in the image?",
            f"Does this image look like: {prompt}?"
        ],
        'choices': [
            ['no', 'yes'],
            ['no', 'yes']
        ],
        'answer': [  
            'yes',   
            'yes' 
        ]
    }
    return qa
    


/root/miniconda3/lib/python3.12/site-packages/spacy/util.py:922: UserWarning: [W095] Model 'en_core_web_lg' (3.7.1) was trained with spaCy v3.7.2 and may not be 100% compatible with the current version (3.8.7). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


In [8]:
print(generate_qa_spacy('crimson rectangles forming a chaotic grid'))

{'question': ['Are there a chaotic grid in the image? Answer yes or no.', 'Are there crimson rectangles in the image? Answer yes or no.', 'Are there crimson rectangles forming a chaotic grid? Answer yes or no.'], 'choices': [['no', 'yes'], ['no', 'yes'], ['no', 'yes']], 'answer': ['yes', 'yes', 'yes']}


## load utility

In [9]:
# %%writefile dllm_utils.py
# import re
# import vtracer
# from IPython.display import SVG, display, Image
# from PIL import Image as PILImage

# def convert_paths_to_polygons(svg_code: str, size: int) -> str:
#     # Find all path elements
#     scale_factor = 384 / size

#     path_pattern = re.compile(
#         r'<path[^>]*d="([^"]+)"[^>]*fill="([^"]+)"[^>]*transform="translate\(([^)]+)\)"[^>]*/?>'
#     )
#     paths = path_pattern.findall(svg_code)

#     polygons = []

#     for d_content, fill_color, translate in paths:
#         tx, ty = map(float, translate.split(','))
#         points = []
#         tokens = re.findall(r'[MLZmlz]|-?\d+\.?\d*,\-?\d+\.?\d*', d_content.strip())
#         for token in tokens:
#             if token in {'M', 'L', 'Z', 'm', 'l', 'z'}:
#                 continue
#             x_str, y_str = token.split(',')
#             x = int(round(float(x_str) + tx))
#             y = int(round(float(y_str) + ty))
#             points.append(f"{x},{y}")

#         if points:
#             polygon = f'<polygon points="{" ".join(points)}" fill="{fill_color}"/>'
#             polygons.append(polygon)

#     # Build the compact SVG
#     new_svg = (
#         f'<svg width="384" height="384" viewBox="0 0 384 384"><g transform="scale({scale_factor})">'
#         + "".join(polygons)
#         + '</g></svg>'
#     )

#     return new_svg


# def fix_svg_size(svg_code: str, target_size: int = 384) -> str:
#     """
#     Update the <svg> tag's width and height to match target size.
#     """
#     # Replace width attribute
#     svg_code = re.sub(
#         r'(width\s*=\s*")([^"]+)(")',
#         lambda m: f'{m.group(1)}{target_size}{m.group(3)}',
#         svg_code,
#         count=1
#     )
#     # Replace height attribute
#     svg_code = re.sub(
#         r'(height\s*=\s*")([^"]+)(")',
#         lambda m: f'{m.group(1)}{target_size}{m.group(3)}',
#         svg_code,
#         count=1
#     )
#     return svg_code


# def bitmap_to_svg_layered(img, input_path, output_path, resolution):
#     default_svg = """<svg width="384" height="384" viewBox="0 0 384 384"><circle cx="50" cy="50" r="40" fill="red" /></svg>"""

#     # Step 1: Resize the input image to 256x256
#     # 256x256 is helpful bc each length od path is shorter, so with the same max_svg_length, we can have more paths(more color)
#     size = resolution
#     img = img.resize((size,size), PILImage.LANCZOS)
#     img.save(input_path)

#     # Step 2: Convert the resized image to SVG
#     max_svg_length = 9996  # target length limit

#     # Define injection to prevent OCR hallucination
#     injection = ''
#     injection_a1 = '<path d="M20 364 L24 356 L28 364 M22 360 L26 360" stroke="#CCCCCC"/>'      # Bottom-left A
#     injection_a2 = '<path d="M364 28 L360 20 L356 28 M362 24 L358 24" stroke="#888888"/>'         # Top-right A
#     injection = injection_a1 + injection_a2
#     injection_length = len(injection)

#     # Binary search for best layer_difference
#     low = 1
#     high = 200
#     best_svg_code = None
#     best_layer_difference = 10

#     try:
#         while low <= high:
#             layer_difference = (low + high) // 2
    
#             vtracer.convert_image_to_svg_py(
#                 input_path,
#                 output_path,
#                 colormode='color',        # Options: 'color' or 'binary'
#                 hierarchical='stacked',   # Options: 'stacked' or 'cutout'
#                 mode='polygon',           # Options: 'spline', 'polygon', or 'none'
#                 filter_speckle=3,          # remove tiny regions
#                 color_precision=8,         # reduce color complexity
#                 layer_difference=layer_difference,  # more aggressive merging
#                 corner_threshold=10,       # remove subtle corners
#                 length_threshold=10,       # remove short paths
#                 max_iterations=10,         # faster, less detail
#                 splice_threshold=10,       # simplify curves
#                 path_precision=3           # reduce vertex detail
#             )
    
#             # Step 3: Read and display the SVG
#             try:
#                 with open(output_path, "r", encoding="utf-8") as f:
#                     svg_code = f.read()
#             except UnicodeDecodeError:
#                 # Bad SVG output (probably corrupted), try again
#                 high = layer_difference - 1
#                 continue  # go back to binary search
    
#             # Clean the first two lines if present
#             lines = svg_code.splitlines()
#             removed_length = 0
#             if lines and lines[0].strip().startswith('<?xml'):
#                 removed_length += len(lines[0]) + 1  # +1 for newline
#                 lines = lines[1:]
#             if lines and lines[0].strip().startswith('<!--'):
#                 removed_length += len(lines[0]) + 1  # +1 for newline
#                 lines = lines[1:]
#             svg_code = "\n".join(lines)
    
#             svg_code = svg_code.replace(
#                 '<svg ',
#                 '<svg viewBox="0 0 384 384" ', #this is 20 bytes more
#                 1  # only replace first occurrence
#             )
            
#             # remove version and xmlns attributes
#             svg_code = re.sub(r'\s*version="[^"]*"', '', svg_code)
#             svg_code = re.sub(r'\s*xmlns="[^"]*"', '', svg_code)
    
#             # use polygon instead of path
#             svg_code = convert_paths_to_polygons(svg_code, size)
                
#             # Correct length check: give credit for removed lines
#             if len(svg_code) + injection_length <= max_svg_length:
#                 best_svg_code = svg_code
#                 best_layer_difference = layer_difference
#                 high = layer_difference - 1  # search for even more detail
#             else:
#                 low = layer_difference + 1  # simplify more


#         # ============= if layer_diff = 200 is not enough, keep increasing ==========================================
#         if best_svg_code == None:
#             layer_difference = 210
#             max_diff = 1000 
#             while layer_difference <= max_diff:
#                 vtracer.convert_image_to_svg_py(
#                     input_path,
#                     output_path,
#                     colormode='color',        # Options: 'color' or 'binary'
#                     hierarchical='stacked',   # Options: 'stacked' or 'cutout'
#                     mode='polygon',           # Options: 'spline', 'polygon', or 'none'
#                     filter_speckle=3,          # remove tiny regions
#                     color_precision=8,         # reduce color complexity
#                     layer_difference=layer_difference,  # more aggressive merging
#                     corner_threshold=10,       # remove subtle corners
#                     length_threshold=10,       # remove short paths
#                     max_iterations=10,         # faster, less detail
#                     splice_threshold=10,       # simplify curves
#                     path_precision=3           # reduce vertex detail
#                 )
        
#                 try:
#                     with open(output_path, "r", encoding="utf-8") as f:
#                         svg_code = f.read()
#                 except UnicodeDecodeError:
#                     # Bad SVG output (probably corrupted), try again
#                     continue  # go back to start of while loop
        
#                 # Clean the first two lines if present
#                 lines = svg_code.splitlines()
#                 removed_length = 0
#                 if lines and lines[0].strip().startswith('<?xml'):
#                     removed_length += len(lines[0]) + 1  # +1 for newline
#                     lines = lines[1:]
#                 if lines and lines[0].strip().startswith('<!--'):
#                     removed_length += len(lines[0]) + 1  # +1 for newline
#                     lines = lines[1:]
#                 svg_code = "\n".join(lines)
        
#                 svg_code = svg_code.replace(
#                     '<svg ',
#                     '<svg viewBox="0 0 384 384" ', #this is 20 bytes more
#                     1  # only replace first occurrence
#                 )
                
#                 # remove version and xmlns attributes
#                 svg_code = re.sub(r'\s*version="[^"]*"', '', svg_code)
#                 svg_code = re.sub(r'\s*xmlns="[^"]*"', '', svg_code)
        
#                 # use polygon instead of path
#                 svg_code = convert_paths_to_polygons(svg_code, size)
#                 print(f'{len(svg_code)=}')
                    
#                 # Correct length check: give credit for removed lines
#                 if len(svg_code) + injection_length <= max_svg_length: # acount for injection
#                     best_svg_code = svg_code
#                     best_layer_difference = layer_difference
#                     break
#                 else:
#                     layer_difference += 10
    
#         svg_code = fix_svg_size(best_svg_code, target_size=384) # doesnt change length
#         # Inject fake letter path to prevent OCR hallucination
#         svg_code = svg_code.replace("</svg>", injection + "</svg>")
#         print(f'{best_layer_difference=}')

#     except Exception as e:
#         print(f"{e}")
#         svg_code = default_svg

    
#     return svg_code

# # ================== color richfullness =========================
# import os
# from PIL import Image
# import numpy as np
# from skimage.color import rgb2lab
# from scipy.stats import entropy
# import matplotlib.pyplot as plt
# def compute_color_richness_entropy(image, bins=32):
#     # Ensure image is RGB and resized
#     image = image.convert('RGB')
#     image = image.resize((256, 256))
#     img_array = np.array(image) / 255.0
#     lab_image = rgb2lab(img_array)

#     # Flatten LAB channels
#     L = lab_image[:, :, 0].flatten()
#     A = lab_image[:, :, 1].flatten()
#     B = lab_image[:, :, 2].flatten()

#     # Compute histograms
#     L_hist, _ = np.histogram(L, bins=bins, density=True)
#     A_hist, _ = np.histogram(A, bins=bins, density=True)
#     B_hist, _ = np.histogram(B, bins=bins, density=True)

#     # Calculate entropy for each channel
#     L_entropy = entropy(L_hist + 1e-8)
#     A_entropy = entropy(A_hist + 1e-8)
#     B_entropy = entropy(B_hist + 1e-8)

#     total_entropy = L_entropy + A_entropy + B_entropy

#     # Normalize: range from 0-10.39
#     max_entropy = 10.39 # = 3 * np.log(bins)
#     normalized_entropy = total_entropy / max_entropy
    
#     return normalized_entropy

# def map_score_to_range(normalized_score):
#     if normalized_score <= 0.5:
#         return 384
#     elif normalized_score >= 0.7:
#         return 128
#     else:
#         # Linear interpolation between 384 and 128
#         alpha = (normalized_score - 0.5) / (0.7 - 0.5)
#         return int(384 - alpha * (384 - 128))


In [10]:
#| export

import random
import re
from colorsys import rgb_to_hls, hls_to_rgb

def add_ocr_decoy_svg(svg_code: str) -> str:
    """
    Adds nested circles with second darkest and second brightest colors from the existing SVG,
    positioned in one of the four corners (randomly selected) but positioned to avoid being
    cropped out during image processing.
    
    Parameters:
    -----------
    svg_code : str
        The original SVG string
    
    Returns:
    --------
    str
        Modified SVG with the nested circles added
    """
    modified_svg = svg_code
    try:        
        width, height = 384, 384
        
        # Function to convert hex color to RGB
        def hex_to_rgb(hex_color):
            hex_color = hex_color.lstrip('#')
            if len(hex_color) == 3:
                hex_color = ''.join([c*2 for c in hex_color])
            return tuple(int(hex_color[i:i+2], 16)/255 for i in (0, 2, 4))
        
        # Function to convert RGB to hex
        def rgb_to_hex(rgb):
            return '#{:02x}{:02x}{:02x}'.format(
                int(rgb[0] * 255), 
                int(rgb[1] * 255), 
                int(rgb[2] * 255)
            )
        
        # Function to calculate color lightness
        def get_lightness(color):
            # Handle different color formats
            if color.startswith('#'):
                rgb = hex_to_rgb(color)
                return rgb_to_hls(*rgb)[1]  # Lightness is the second value in HLS
            elif color.startswith('rgb'):
                rgb_match = re.search(r'rgb\((\d+),\s*(\d+),\s*(\d+)\)', color)
                if rgb_match:
                    r, g, b = map(lambda x: int(x)/255, rgb_match.groups())
                    return rgb_to_hls(r, g, b)[1]
            return 0.5  # Default lightness if we can't parse
        
        # Extract all colors from the SVG
        color_matches = re.findall(r'(?:fill|stroke)="(#[0-9A-Fa-f]{3,6}|rgb\(\d+,\s*\d+,\s*\d+\))"', svg_code)
        
        # Default colors in case we don't find enough
        second_darkest_color = "#333333"  # Default to dark gray
        second_brightest_color = "#CCCCCC"  # Default to light gray
        
        if color_matches:
            # Remove duplicates and get unique colors
            unique_colors = list(set(color_matches))
            
            # Calculate lightness for each unique color
            colors_with_lightness = [(color, get_lightness(color)) for color in unique_colors]
            
            # Sort by lightness (brightness)
            sorted_colors = sorted(colors_with_lightness, key=lambda x: x[1])
            
            # Handle different scenarios based on number of unique colors
            if len(sorted_colors) >= 4:
                # We have at least 4 unique colors - use 2nd darkest and 2nd brightest
                second_darkest_color = sorted_colors[1][0]
                second_brightest_color = sorted_colors[-2][0]
            elif len(sorted_colors) == 3:
                # We have 3 unique colors - use 2nd darkest and brightest
                second_darkest_color = sorted_colors[1][0]
                second_brightest_color = sorted_colors[2][0]
            elif len(sorted_colors) == 2:
                # We have only 2 unique colors - use the darkest and brightest
                second_darkest_color = sorted_colors[0][0]
                second_brightest_color = sorted_colors[1][0]
            elif len(sorted_colors) == 1:
                # Only one color - use it for second_darkest and a derived lighter version
                base_color = sorted_colors[0][0]
                base_lightness = sorted_colors[0][1]
                second_darkest_color = base_color
                
                # Create a lighter color variant if the base is dark, or darker if base is light
                if base_lightness < 0.5:
                    # Base is dark, create lighter variant
                    second_brightest_color = "#CCCCCC"
                else:
                    # Base is light, create darker variant
                    second_darkest_color = "#333333"
        
        # Ensure the colors are different
        if second_darkest_color == second_brightest_color:
            # If they ended up the same, modify one of them
            if get_lightness(second_darkest_color) < 0.5:
                # It's a dark color, make the bright one lighter
                second_brightest_color = "#CCCCCC"
            else:
                # It's a light color, make the dark one darker
                second_darkest_color = "#333333"
        
        # Base size for the outer circle
        base_outer_radius = width * 0.023
        
        # Randomize size by ±10%
        size_variation = base_outer_radius * 0.1
        outer_radius = base_outer_radius + random.uniform(-size_variation, size_variation)
        
        # Define radii for inner circles based on outer radius
        middle_radius = outer_radius * 0.80
        inner_radius = middle_radius * 0.65
        
        # Calculate the maximum crop margin based on the image processing (5% of dimensions)
        # Add 20% extra margin for safety
        crop_margin_w = int(width * 0.05 * 1.2)
        crop_margin_h = int(height * 0.05 * 1.2)
        
        # Calculate center point based on the outer radius to ensure the entire circle stays visible
        safe_offset = outer_radius + max(crop_margin_w, crop_margin_h)
        
        # Choose a random corner (0: top-left, 1: top-right, 2: bottom-left, 3: bottom-right)
        corner = random.randint(0, 3)
        
        # Position the circle in the chosen corner, accounting for crop margin
        if corner == 0:  # Top-left
            center_x = safe_offset
            center_y = safe_offset
        elif corner == 1:  # Top-right
            center_x = width - safe_offset
            center_y = safe_offset
        elif corner == 2:  # Bottom-left
            center_x = safe_offset
            center_y = height - safe_offset
        else:  # Bottom-right
            center_x = width - safe_offset
            center_y = height - safe_offset
        
        # Add a small random offset (±10% of safe_offset) to make positioning less predictable
        random_offset = safe_offset * 0.1
        center_x += random.uniform(-random_offset, random_offset)
        center_y += random.uniform(-random_offset, random_offset)
        
        # Round to 1 decimal place to keep file size down
        outer_radius = int(outer_radius)
        middle_radius = int(middle_radius)
        inner_radius = int(inner_radius)
        center_x = int(center_x)
        center_y = int(center_y)
        
        # Create the nested circles
        outer_circle = f'<circle cx="{center_x}" cy="{center_y}" r="{outer_radius}" fill="{second_darkest_color}" />'
        middle_circle = f'<circle cx="{center_x}" cy="{center_y}" r="{middle_radius}" stroke="{second_brightest_color}" fill="none"/>'
        # inner_circle = f'<circle cx="{center_x}" cy="{center_y}" r="{inner_radius}" fill="{second_darkest_color}" />'
        
        # Create a group element that contains all three circles
        # group_element = f'{outer_circle}{middle_circle}{inner_circle}'
        group_element = f'{outer_circle}{middle_circle}'
        
        # Insert the group element just before the closing SVG tag
        modified_svg = svg_code.replace("</svg>", f"{group_element}</svg>")
        
        # Calculate and add a comment with the byte size information
        outer_bytes = len(outer_circle.encode('utf-8'))
        middle_bytes = len(middle_circle.encode('utf-8'))
        # inner_bytes = len(inner_circle.encode('utf-8'))
        total_bytes = outer_bytes + middle_bytes 
        print(total_bytes)
        
    except Exception as e:
        print(e)
    return modified_svg

## load mp workers

In [11]:
# %%writefile workers.py
# import os, io, ast, re, gc, time, math, string, statistics
# from io import BytesIO
# from multiprocessing import Process, Manager
# import numpy as np, pandas as pd, matplotlib.pyplot as plt, cv2
# import torch, torch.nn as nn
# from PIL import Image as PILImage, ImageFilter
# from more_itertools import chunked
# import cairosvg, vtracer, clip

# from transformers import AutoProcessor, PaliGemmaForConditionalGeneration, T5EncoderModel, BitsAndBytesConfig
# from diffusers import (
#     DiffusionPipeline, DPMSolverSinglestepScheduler, DDIMScheduler,
#     AutoencoderKL, AutoencoderTiny, FluxTransformer2DModel, FluxPipeline,
#     BitsAndBytesConfig as DiffusersBitsAndBytesConfig, GGUFQuantizationConfig
# )
# from diffusers.hooks import apply_group_offloading

# from dllm_utils import map_score_to_range, compute_color_richness_entropy, bitmap_to_svg_layered
# from metricMP import VQAEvaluator, AestheticEvaluator, AestheticPredictor
# import metricMP as metric

# input_path = "output.png"
# output_path = "output.svg"

# negative_prompt = (
#     "text, logo, mirror reflection, high-reflective, lines, deformed, ugly, "
#     "wrong proportion, low res, bad anatomy, worst quality, low quality, "
#     "framing, hatching, patterns, outlines"
# )

# # ============================= diffusion function ==================================
# # ——— 1) Placeholders for model paths & their names ———
# _path0 = None
# _name0 = None

# # ——— 1b) Placeholders for aux checkpoints needed by your loaders ———
# _lora_path    = None
# _hypersd_path = None
# _t5_path      = None
# _flux_path    = None
# _vae_path     = None

# # ——— 2) Shared negative prompt ———
# negative_prompt = (
#     'text, logo, mirror reflection, high-reflective, lines, deformed, ugly, '
#     'wrong proportion, low res, bad anatomy, worst quality, low quality, '
#     'framing, hatching, patterns, outlines'
# )

# # ============================= diffusion function ==================================
# _pipe0 = None
# _name0 = None

# def set_pipelines(pipe0, name0):
#     """
#     Inject the two CPU-loaded pipelines and their model names.
#     """
#     global _pipe0,_name0
#     _pipe0, _name0 = pipe0, name0

# _gen_configs = {}

# def set_gen_configs(configs: dict):
#     global _gen_configs
#     _gen_configs = configs


# # ============================= evaluator function ==================================
# _pali_model = None
# _pali_processor = None
# def set_palimodel(pali_model, pali_processor):
#     global _pali_model, _pali_processor
#     _pali_model = pali_model
#     _pali_processor = pali_processor
    

# # ============================= helper function ==================================

# def log(msg):
#     with open("worker.log", "a") as f:
#         f.write(f"[{time.strftime('%H:%M:%S')}] {msg}\n")
#         f.flush()
        
# def load(device):
#         model_path = 'sac-logos-ava1-l14-linearmse/sac+logos+ava1-l14-linearMSE.pth'
#         clip_model_path = 'clip-vit-large-patch14/ViT-L-14.pt'
#         state_dict = torch.load(model_path, weights_only=True, map_location=device)
        
#         # CLIP embedding dim is 768 for CLIP ViT L 14
#         predictor = AestheticPredictor(768)
#         predictor.load_state_dict(state_dict)
#         predictor.to(device)
#         predictor.eval()
#         clip_model, preprocessor = clip.load(clip_model_path, device=device)
        
#         return predictor, clip_model, preprocessor

# def image_resize(image, size=(384, 384)):
#     return image.convert('RGB').resize(size)

# def get_score(sample, qa, aesthetic_evaluator, vqa_evaluator, vqa = True,):

#     try:
#         # If sample is a string, treat as SVG and convert to image
#         rng = np.random.RandomState(42)
#         group_seed = rng.randint(0, np.iinfo(np.int32).max)
#         if isinstance(sample, str):
#             image = metric.svg_to_png(sample)
#         else:
#             image = sample
        
#         image_processor = metric.ImageProcessor(image=image_resize(image), seed=group_seed).apply()
#         image = image_processor.image.copy()
    
#         try:
#             aesthetic_score = aesthetic_evaluator.score(image)
#         except Exception as e:
#             print(f"AES score error: {e}")
#             aesthetic_score = 0.5

#         if vqa:
#             try:
#                 questions = qa['question']
#                 choices = qa['choices']
#                 answers = qa['answer']
#                 vqa_score = vqa_evaluator.score(questions, choices, answers, image)
#             except Exception as e:
#                 print(f"VQA score error: {e}")
#                 raise
                
#                 # vqa_score = 0.5
#         else:
#             vqa_score = 0.5
            
#         ocr_score = 1.0
    
#         instance_score = metric.harmonic_mean(vqa_score, aesthetic_score, beta=0.5) * ocr_score
    
#         return instance_score, aesthetic_score, ocr_score, vqa_score
    
#     except Exception as e:
#         print(f"score error: {e}")
#         raise
#         # print(f"score error: {e}")
#         # return 0.5, 0.5, 1.0, 0.5

        
# def drain_queue(q):
#     try:
#         while True:
#             q.get_nowait()
#     except:
#         pass
        
# # ============================= main function ==================================
# def _generator_loop(pipe0, name0, gen_configs,
#                     prompt_q, image_q, rank):
#     # rank == 1
#     torch.cuda.set_device(rank)

#     pipe      = pipe0
#     model_name = name0
    
#     pipe = pipe.to(rank)
#     if model_name in ('flash','sdxl'):
#         pipe.unet = torch.compile(pipe.unet, backend="eager", fullgraph=True)
#     torch.cuda.empty_cache()
    
#     while True:
#         item = prompt_q.get()
#         if item is None:
#             break
#         tag, prompt, qa, attempts = item
#         drain_queue(image_q)
#         cfg = gen_configs[model_name]
#         for i in range(attempts):
#             start = time.time()
#             imgs = pipe(prompt, negative_prompt=negative_prompt, **cfg).images
#             for bitmap in imgs:
#                 color_score = compute_color_richness_entropy(bitmap)
#                 resolution = map_score_to_range(color_score)
#                 # bitmap → SVG
#                 svg = bitmap_to_svg_layered(bitmap, input_path, output_path, resolution)
#                 buf = BytesIO()
#                 bitmap = bitmap.resize((384,384))
#                 bitmap.save(buf, format="PNG")
#                 end = time.time()
#                 # log(f'image {i} takes {end - start:.2f}s to produce!')
#                 image_q.put((tag, buf.getvalue(), svg, qa))
            

# def _scorer_loop(image_q, result_q, pali_model, pali_processor, rank):
#     # rank == 0
#     torch.cuda.set_device(rank)

#     # pali_model = pali_model.to(f'cuda:{rank}')
#     # create instance
#     vqa_evaluator = VQAEvaluator(pali_model, pali_processor)
    
#     # load aes model =============
#     aes_predictor, clip_model, aes_preprocessor = load(f'cuda:{rank}')
#     aesthetic_evaluator = AestheticEvaluator(aes_predictor, clip_model, aes_preprocessor, f'cuda:{rank}')

#     from io import BytesIO
#     while True:
#         item = image_q.get()
#         if item is None:
#             break
#         tag, png_byte, svg, qa = item
#         start = time.time()
#         # VQA/aesthetic scoring
#         instance_score, aesthetic_score, ocr_score, vqa_score = get_score(sample=svg, qa=qa, aesthetic_evaluator= aesthetic_evaluator, vqa_evaluator = vqa_evaluator)
#         end = time.time()
#         # log(f'image takes {end - start:.2f}s to score!')
#         result_q.put((tag, instance_score, aesthetic_score, ocr_score, vqa_score, svg, png_byte))
#         gc.collect(); torch.cuda.empty_cache()

# def start_workers():
#     mgr = Manager()
#     prompt_q = mgr.Queue()
#     image_q = mgr.Queue()
#     result_q = mgr.Queue()

#     p0 = Process(target=_scorer_loop, args=(image_q, result_q, _pali_model, _pali_processor ,0), daemon=True)
#     p1 = Process(target=_generator_loop, args=(_pipe0, _name0, _gen_configs,
#                                                prompt_q, image_q, 1), daemon=True)

#     p0.start()
#     p1.start()

#     return prompt_q, result_q, [p0, p1], image_q

# def stop_workers(prompt_q, image_q, procs):
#     prompt_q.put(None)
#     image_q.put(None)
#     for p in procs:
#         p.join()

## load SD

In [12]:
import workers
from workers import start_workers, stop_workers

import torch
from diffusers import DiffusionPipeline, DPMSolverSinglestepScheduler, DDIMScheduler
from diffusers import FluxTransformer2DModel
from diffusers import BitsAndBytesConfig as DiffusersBitsAndBytesConfig, FluxTransformer2DModel, FluxPipeline, GGUFQuantizationConfig
from transformers import BitsAndBytesConfig as BitsAndBytesConfig, T5EncoderModel
from diffusers import AutoencoderKL, AutoencoderTiny
# from diffusers.hooks import apply_group_offloading
import gc

# path for sdxl
flash_path    = 'sdxl-flash'
lora_path     = 'lora'

def load_flash():
    base = DiffusionPipeline.from_pretrained(flash_path, torch_dtype=torch.float16, use_safetensors=True)
    base.scheduler = DPMSolverSinglestepScheduler.from_config(base.scheduler.config, timestep_spacing="trailing")
    
    base.load_lora_weights(lora_path, weight_name='Vector_illustration_XL.safetensors')
    base.fuse_lora(lora_scale=0.8)
    return base

gen_configs = {
  'flash': dict(width=768, height=768, num_inference_steps=7, guidance_scale=2.5, num_images_per_prompt=1),
}


In [13]:
#| export
import workers
from workers import start_workers, stop_workers
base, name = load_flash(), 'flash'

workers.set_pipelines(base, name)
workers.set_gen_configs(gen_configs)


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

## load pali

In [14]:
#| export
import time
from transformers import AutoProcessor, PaliGemmaForConditionalGeneration, T5EncoderModel, BitsAndBytesConfig
start = time.time()
# load paligemma2 =============
quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type='nf4',
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16,
        )
# quantization_config = BitsAndBytesConfig(load_in_8bit=True) # faster than 4bit but still fit in 15gb vram
pali_model_path = 'paligemma2-10b-mix-448'
pali_processor = AutoProcessor.from_pretrained(pali_model_path)
pali_model = PaliGemmaForConditionalGeneration.from_pretrained(
    pali_model_path,
    low_cpu_mem_usage=True,
    quantization_config=quantization_config,
    device_map = 0
)
end = time.time()
print(f'loading takes {end-start:.2f}s')
workers.set_palimodel(pali_model, pali_processor)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

loading takes 8.46s


## Implement the package Model class

In [15]:
#| export
import time, queue, torch
from PIL import Image
from io import BytesIO

class Model:
    def __init__(self):
        # fallback SVG
        self.default_svg = (
            '<svg width="384" height="384" viewBox="0 0 384 384">'
            '<circle cx="50" cy="50" r="40" fill="red"/></svg>'
        )

        # **RESTORE** your prompt formatting
        self.prompt_prefix = "a stylized digital painting presenting a"
        self.prompt_suffix = (
            ". The painting promote vector-art aesthetic, "
            "in watercolor art style with vibrant and clean background. "
            "The overall atmosphere is tranquil yet powerful, raw-photo "
            "hyper-detail, 4K, cinematic lighting, award-winning, masterpiece."
        )

        self._workers_started = False
        self.attempt_0 = 8
        self.attempt_1 = 2

    def _ensure_workers(self):
        if not self._workers_started:
            self.prompt_q, self.result_q, self.procs, self.image_q = start_workers()
            self._workers_started = True

    def predict_impl(self, prompt: str):
        start = time.time()
        if not prompt:
            return self.default_svg, None
        
        self._ensure_workers()
        
        qa = generate_qa_spacy(prompt)
        print(f'{qa=}')
        
        prompt_0 = f"{self.prompt_prefix} {prompt}{self.prompt_suffix}"
        prompt_1 = "a simplified color icon presenting " + prompt + (
            ". The graphic promotes vector-art aesthetic, "
            "in watercolor art style with vibrant and clean background. "
            "The overall atmosphere is tranquil yet powerful, raw-photo "
            "hyper-detail, 4K, cinematic lighting, award-winning, masterpiece."
        )
        print(f'============== {prompt_0} =====================')
        print(f'============== {prompt_1} =====================')
        tag = str(time.time_ns())

        # ======================== first attempt =================================
        # dispatch into the two‐GPU pipeline
        self.prompt_q.put((tag, prompt_0, qa, self.attempt_0))

        best_score = 0.0
        best_svg   = self.default_svg
        best_img   = None
        count = 0

        while count < self.attempt_0:
            try:
                item = self.result_q.get()
                print('result get!')
            except queue.Empty:
                break

            # print(item[0])
            t, instance_score, aesthetic_score, ocr, vqa_score, svg, png_bytes = item
            if t != tag:
                continue

            bitmap = Image.open(BytesIO(png_bytes))
            display(bitmap.resize((128,128)))
            score = instance_score
            print(f'{aesthetic_score =}, {vqa_score=}, {score=}')
            print('svg length:', len(svg))
            
            if score > best_score:
                best_score = score
                best_svg   = svg
                best_img   = bitmap
            count += 1

        # ======================== second attempt =================================
        # dispatch into the two‐GPU pipeline
        if best_score < 0.5:
            print(' %%%%%%%%%%%%%%%%%%%%%%%%%%% switch prompt %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%')
            self.prompt_q.put((tag, prompt_1, qa, self.attempt_1))
        else:
            self.prompt_q.put((tag, prompt_0, qa, self.attempt_1))

        while count < self.attempt_0 + self.attempt_1:
            try:
                item = self.result_q.get()
                print('result get!')
            except queue.Empty:
                break

            # print(item[0])
            t, instance_score, aesthetic_score, ocr, vqa_score, svg, png_bytes = item
            if t != tag:
                continue

            bitmap = Image.open(BytesIO(png_bytes))
            display(bitmap.resize((128,128)))
            score = instance_score
            print(f'{aesthetic_score =}, {vqa_score=}, {score=}')
            print('svg length:', len(svg))
            
            if score > best_score:
                best_score = score
                best_svg   = svg
                best_img   = bitmap
            count += 1
   
        end = time.time()
        print(f'==========={count} images in total, takes {end - start:.2f}s==================')
        print(f'=========== best score: {best_score}==================')
        print(f'=========== svg length: {len(best_svg)}==================')
        return best_svg, best_img

    def predict(self, prompt: str):
        svg, _ = self.predict_impl(prompt)
        torch.cuda.empty_cache()
        return svg

    def close(self):
        stop_workers(self.prompt_q, self.procs)


In [16]:
model = Model()

In [17]:
# %%time
# r = train_df.iloc[0]
# description = r.description
# # print(description)
# svg, img = model.predict_impl(description)
# display(img)
# display(SVG(svg))
# print(svg)

In [18]:
# SVG_check = kagglehub.package_import('metric/svg-constraints/versions/1')
# constraints = SVG_check.SVGConstraints()
# constraints.validate_svg(svg) 

## validation

In [19]:
from tqdm.notebook import tqdm
tqdm.pandas()  # enables .progress_apply()

In [20]:
import matplotlib.pyplot as plt
%matplotlib inline
import pandas as pd
from tqdm.auto import tqdm
tqdm.pandas()
import ast
df = pd.read_csv("train_set.csv")
df = df.sample(n=5, random_state=33).reset_index(drop=True)
df_parsed = df.copy()
df['multiple_choice_qa'] = df_parsed['multiple_choice_qa'].apply(ast.literal_eval)
df


,id,description,multiple_choice_qa
0,5a74f8ac,a metallic blue sphere to the left of a brown ...,"{'question': ['What color is the sphere?', 'Wh..."
1,bef01b35,The Great Pyramid of Giza situated in front of...,{'question': ['Is the Great Pyramid of Giza in...
2,0b948200,slices of avocado on a piece of toast,{'question': ['What is on the piece of toast?'...
3,57c99708,a sword slicing through a loaf of bread,{'question': ['Is there a sword in the image?'...
4,7ecf9e03,a horned owl with a graduation cap and diploma,"{'question': ['Is there an owl in the image?',..."


In [21]:
%%capture cap
df['svg'] = df.description.progress_apply(model.predict)
df.to_csv('res_sd.csv')

Changing scheduler {self.config} to have `lower_order_final` set to True to handle uneven amount of inference steps. Please make sure to always use an even number of `num_inference steps when using `lower_order_final=False`.
100%|██████████| 7/7 [00:01<00:00,  5.93it/s]


best_layer_difference=30


100%|██████████| 7/7 [00:00<00:00,  7.12it/s]


best_layer_difference=36


100%|██████████| 7/7 [00:00<00:00,  7.14it/s]


...

100%|██████████| 7/7 [00:01<00:00,  6.81it/s]


best_layer_difference=111


100%|██████████| 7/7 [00:01<00:00,  6.72it/s]


best_layer_difference=106


100%|██████████| 7/7 [00:01<00:00,  6.84it/s]
